# Downstream tasks

In [ ]:
import sys
from pathlib import Path

import scanpy as sc

import warnings

warnings.filterwarnings("ignore")

In [ ]:
from interscale.evaluation.downstream_classification import (
    classify,
    feature_sets_from_spec,
    summarize,
)
from interscale.evaluation.downstream_regression import (
    attention_pairs,
    regress_attention,
    residual_flow,
    score_against_truth,
)
from interscale.evaluation.synthetic_data import make_synthetic

In [ ]:
BASE_DIR_PROJECT = Path.cwd().resolve().parent.parent

sys.path.insert(0, str(BASE_DIR_PROJECT))
DATA = "synth_data_0"

## Load data

In [ ]:
adata = sc.read_h5ad(f"{BASE_DIR_PROJECT}/data/{DATA}_trained.h5ad")
adata

## Classification

In [ ]:
res = classify(adata, "cell_type", level="node", sample_key="slide", group_key="donor", layer="log1p_norm")
summarize(res)

In [ ]:
res_cond = classify(adata, "condition", level="graph", sample_key="slide", group_key="donor", layer="log1p_norm")
summarize(res_cond)

In [ ]:
sets = feature_sets_from_spec(
    adata,
    [
        "combined_local_emb",
        "combined_global_emb",
        "combined_local_emb+combined_global_emb",
        "X_pca",
        "expression",
    ],
)

summarize(classify(adata, "niche", feature_sets=sets, level="node", layer="log1p_norm"))

## Regression

In [ ]:
pairs = attention_pairs(
    adata,
    prefix="combined",
    sample_key="slide",
    obs_features=("cell_type", "niche", "total_counts", "n_genes_by_counts"),
    normalize="graph_z",
    max_pairs=200_000,
)

pairs.head()

In [ ]:
result = regress_attention(pairs, estimator="ridge", group_key="graph", cv_folds=5)
result.variance

In [ ]:
result.nonlinear_r2

In [ ]:
residual_flow(result.pairs)

In [ ]:
score_against_truth(adata, result.pairs)

## Create the synthetic dataset

In [ ]:
adata_synth = make_synthetic()
adata_synth

In [ ]:
adata_synth.write_h5ad(f"{BASE_DIR_PROJECT}/data/{DATA}.h5ad")